# 实验性车辆形式抓取笔记本

用于测试数据状态以及生成形式数据

In [3]:
import httpx
from bs4 import BeautifulSoup
import re
import json

In [5]:
def fetch_wikitext(page_title: str) -> str:
    """获取页面的原始Wikitext"""
    url = "https://ja.wikipedia.org/w/api.php"
    params = {
        "action": "query",
        "titles": page_title,
        "prop": "revisions",
        "rvprop": "content",     # 返回原始wikitext
        "rvslots": "main",
        "format": "json",
    }
    resp = httpx.get(url, params=params)
    pages = resp.json()["query"]["pages"]
    page = next(iter(pages.values()))
    return page["revisions"][0]["slots"]["main"]["*"]

In [11]:
url = "https://ja.wikipedia.org/w/api.php"
params = {
    "action": "query",
    "titles": "JR東日本の車両形式",
    "prop": "revisions",
    "rvprop": "content",     # 返回原始wikitext
    "rvslots": "main",
    "format": "json",
}
headers = {
        # Wikipedia要求格式：工具名称/版本 (联系方式)
        "User-Agent": "JapaneseTrainDatasetBuilder/1.0 (research project; fengyukunfyk@gmail.com)"
    }

resp = httpx.get(url, params=params, headers=headers)
if resp.status_code == 200:
    print(f"页面：{params['titles']}请求成功")
    
pages = resp.json()["query"]["pages"]
page = next(iter(pages.values()))
wikitext = page["revisions"][0]["slots"]["main"]["*"]
wikitext[:500]  # 打印前500字符预览

页面：JR東日本の車両形式请求成功


"{{Pathnav|[[東日本旅客鉄道|東日本旅客鉄道（JR東日本）]]|frame=1}}\n'''JR東日本の車両形式'''（ジェイアールひがしにほんのしゃりょうけいしき）は、[[東日本旅客鉄道]]（JR東日本）に在籍する、あるいは在籍した[[鉄道車両]]の一覧である。\n\n== 現在の所属車両 ==\n=== 新幹線電車 ===\n*'''営業用'''\n**[[新幹線E2系電車|E2系]]\n**[[新幹線E3系電車|E3系]]\n**[[新幹線E5系・H5系電車|E5系]]\n**[[新幹線E6系電車|E6系]]\n**[[新幹線E7系・W7系電車|E7系]]\n**[[新幹線E8系電車|E8系]]\n*'''事業用'''\n**[[新幹線E926形電車|E926形]]（電気・軌道総合検測車）\n**[[新幹線E956形電車|E956形]]（高速試験車）\n\n=== 蒸気機関車 ===\n* [[国鉄C57形蒸気機関車|C57形]] - [[国鉄C57形蒸気機関車180号機|180号機]]\n* [[国鉄C58形蒸気機関車|C58形]] - [[国鉄C58形蒸気機関車#動態保存機|239号機]]{{efn2"

In [ ]:
# 解析车种信息
#页面格式示例如下
""""'''JR東日本の車両形式'''（ジェイアールひがしにほんのしゃりょうけいしき）は、[[東日本旅客鉄道]]（JR東日本）に在籍する、あるいは在籍した[[鉄道車両]]の一覧である。\n",
 '\n',
 '== 現在の所属車両 ==\n',
 '=== 新幹線電車 ===\n',
 "*'''営業用'''\n",
 '**[[新幹線E2系電車|E2系]]\n',
 '**[[新幹線E3系電車|E3系]]\n',
 '**[[新幹線E5系・H5系電車|E5系]]\n',
 '**[[新幹線E6系電車|E6系]]\n',
 '**[[新幹線E7系・W7系電車|E7系]]\n',
 '**[[新幹線E8系電車|E8系]]\n',
 "*'''事業用'''\n",
 '**[[新幹線E926形電車|E926形]]（電気・軌道総合検測車）\n',
 '**[[新幹線E956形電車|E956形]]（高速試験車）\n',
 '\n',
 '=== 電気機関車 ===\n',
 "* '''直流用'''\n",
 '** [[国鉄EF64形電気機関車|EF64形]]\n',
 '** [[国鉄EF65形電気機関車|EF65形]]\n',
 "* '''交直両用'''\n",
 '** [[国鉄EF81形電気機関車|EF81形]]\n',
 '\n',
 '=== ディーゼル機関車 ===\n',
 '* [[国鉄DD14形ディーゼル機関車|DD14形]]{{efn2|name="保留車"}}\n',
 '* [[国鉄DD51形ディーゼル機関車|DD51形]]\n',
 '* [[国鉄DE10形ディーゼル機関車|DE10形]]\n',
 '\n',
 '=== 電車 ===\n',
 "* '''旧形営業用'''\n",
 "** '''直流用'''\n",
 '*** [[国鉄31系電車|クモハ12形]]{{efn2|name="保留車"}}\n',
 "* '''特急形'''\n",
 "** '''直流用'''\n",
"""



import re

def parse_vehicle_wikitext(lines: list[str]) -> list[dict]:
    # 匹配 [[页面名|显示名]] 或 [[页面名]]
    link_re = re.compile(r'\[\[([^\]|]+)(?:\|([^\]]+))?\]\]')
    
    # 车型名格式：字母/片假名/汉字前缀 + 数字 + 可选的系/形
    # 覆盖：E5系、223系、EF64形、キハ40系、C57形、GV-E400系、HB-E210系
    series_re = re.compile(
        r'^[A-Za-z\u30A0-\u30FF\u4E00-\u9FFF]'  # 首字符
        r'[A-Za-z\u30A0-\u30FF\u4E00-\u9FFF\-]*' # 前缀（含连字符，如HB-E）
        r'\d+(?:系|形)?$'                          # 数字结尾，可选系/形
    )

    results = []
    current_h2 = ""
    current_h3 = ""
    skip_sections = {"脚注", "注釈", "出典", "関連項目", "外部リンク"}

    for line in lines:
        line = line.rstrip('\n')

        # 解析H2标题
        m = re.match(r'^== (.+?) ==$', line)
        if m:
            current_h2 = m.group(1)
            current_h3 = ""
            continue

        # 解析H3标题
        m = re.match(r'^=== (.+?) ===$', line)
        if m:
            current_h3 = m.group(1)
            continue

        # 跳过无关章节
        if current_h2 in skip_sections:
            continue

        # 只处理列表行（* 开头）
        if not line.lstrip().startswith('*'):
            continue

        # 不跳过纯粗体的用途标签行（'''営業用''' 等，不含[[），保留一级用途分类，例如营业用。
        
        subtype = re.match(r"\*\s*\'\'\'(.+?)\'\'\'", line.lstrip())
        if subtype:
            current_subtype = subtype.group(1)

        # 提取所有链接
        for m in link_re.finditer(line):
            page  = m.group(1)   # 页面名
            label = m.group(2) or page  # 显示名，没有|时用页面名

            # 去掉显示名里的注释（如 "E5系・H5系" 只取E5系）
            label = label.split('・')[0].split('（')[0].strip()

            if series_re.match(label):
                results.append({
                    "series":    label,
                    "wiki_title": page,
                    "status":        current_h2,   # 現在の所属車両 / 過去の所属車両
                    "type":        current_h3,   # 新幹線電車 / 電車 / 気動車 ...
                    "subtype":     current_subtype
                })
        
        results = [
            {**r, "status": r["status"].replace("現在の所属車両", "現役").replace("過去の所属車両", "廃止")}
            for r in results
        ]  # type: ignore

    return results


In [41]:
wikitext.splitlines("\n")

['{{Pathnav|[[東日本旅客鉄道|東日本旅客鉄道（JR東日本）]]|frame=1}}\n',
 "'''JR東日本の車両形式'''（ジェイアールひがしにほんのしゃりょうけいしき）は、[[東日本旅客鉄道]]（JR東日本）に在籍する、あるいは在籍した[[鉄道車両]]の一覧である。\n",
 '\n',
 '== 現在の所属車両 ==\n',
 '=== 新幹線電車 ===\n',
 "*'''営業用'''\n",
 '**[[新幹線E2系電車|E2系]]\n',
 '**[[新幹線E3系電車|E3系]]\n',
 '**[[新幹線E5系・H5系電車|E5系]]\n',
 '**[[新幹線E6系電車|E6系]]\n',
 '**[[新幹線E7系・W7系電車|E7系]]\n',
 '**[[新幹線E8系電車|E8系]]\n',
 "*'''事業用'''\n",
 '**[[新幹線E926形電車|E926形]]（電気・軌道総合検測車）\n',
 '**[[新幹線E956形電車|E956形]]（高速試験車）\n',
 '\n',
 '=== 蒸気機関車 ===\n',
 '* [[国鉄C57形蒸気機関車|C57形]] - [[国鉄C57形蒸気機関車180号機|180号機]]\n',
 '* [[国鉄C58形蒸気機関車|C58形]] - [[国鉄C58形蒸気機関車#動態保存機|239号機]]{{efn2|かつては[[国鉄C58形蒸気機関車#C58 363|363号機]]も保有した。}}\n',
 '* [[国鉄C61形蒸気機関車|C61形]] - [[国鉄C61形蒸気機関車20号機|20号機]]\n',
 '* [[国鉄D51形蒸気機関車|D51形]] - [[国鉄D51形蒸気機関車498号機|498号機]]\n',
 '\n',
 '=== 電気機関車 ===\n',
 "* '''直流用'''\n",
 '** [[国鉄EF64形電気機関車|EF64形]]\n',
 '** [[国鉄EF65形電気機関車|EF65形]]\n',
 "* '''交直両用'''\n",
 '** [[国鉄EF81形電気機関車|EF81形]]\n',
 '\n',
 '=== ディーゼル機関車 ===\n',
 '* [[国鉄DD14形ディーゼル機関車|DD14形]]{{efn2|nam

In [43]:
all_series = parse_vehicle_wikitext(wikitext.splitlines("\n"))
all_series

[{'series': 'E2系',
  'wiki_title': '新幹線E2系電車',
  'status': '現役',
  'type': '新幹線電車',
  'subtype': ''},
 {'series': 'E3系',
  'wiki_title': '新幹線E3系電車',
  'status': '現役',
  'type': '新幹線電車',
  'subtype': ''},
 {'series': 'E5系',
  'wiki_title': '新幹線E5系・H5系電車',
  'status': '現役',
  'type': '新幹線電車',
  'subtype': ''},
 {'series': 'E6系',
  'wiki_title': '新幹線E6系電車',
  'status': '現役',
  'type': '新幹線電車',
  'subtype': ''},
 {'series': 'E7系',
  'wiki_title': '新幹線E7系・W7系電車',
  'status': '現役',
  'type': '新幹線電車',
  'subtype': ''},
 {'series': 'E8系',
  'wiki_title': '新幹線E8系電車',
  'status': '現役',
  'type': '新幹線電車',
  'subtype': ''},
 {'series': 'E926形',
  'wiki_title': '新幹線E926形電車',
  'status': '現役',
  'type': '新幹線電車',
  'subtype': ''},
 {'series': 'E956形',
  'wiki_title': '新幹線E956形電車',
  'status': '現役',
  'type': '新幹線電車',
  'subtype': ''},
 {'series': 'C57形',
  'wiki_title': '国鉄C57形蒸気機関車',
  'status': '現役',
  'type': '蒸気機関車',
  'subtype': ''},
 {'series': 'C58形',
  'wiki_title': '国鉄C58形蒸気機関車',
  'status':